### Middleware
###### middleware provides a way to more tightly control what happens inside the agent
###### it is a function that takes in a list of messages and returns a list of messages
###### it can be used to add logging, add a wait time, or add a custom function
###### it can be used to transform prompts , tool selection and output formatting
###### adding retry logic, rate limiting, and early termination logic 
###### apply guardrails, pii detection, and other policies


In [13]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROK_API_KEY")

###### builtin middlewares: - summarization , human in the feedback , model calling limit

#### summarization middleware 
###### automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older contexts , Summarization is useful for the following :


 - long running conversations that exceed context winodows
 - multi - turn dialouges with extensive historu
 - applications where preserving full conversation context matters

 ###### context window is the llms working memory , the maximum amount of text measured in tokens that an llm can process and reference at a time.

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### message based summarization
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
    system_prompt=SystemMessage(
        content="You are a helpful assistant that can summarize conversations."
    ),
)

In [ ]:
### Run with thread id
config ={"configurable":{"thread_id":"123"}} #SAME CONVERSATION ACROSS TURNS 

In [ ]:
#alternative test data
questions=[
    "what is 2+2?",
    "what is 5*8?",
    "what is 100-3?",
    "what is 64/2?",
    "what is 10%2?",
    "what is 10**2?", 
    "what is 10//4?",
    "what is 10%3?",
]

for question in questions:
    response = agent.invoke({"messages":[HumanMessage(content=question)]},config=config)
    print(f"message: {response}")
    print(f"message:: {len(response['messages'])}")


message: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='95cb5c6c-8dd0-4c56-9b30-6c3de82d1b1e'), AIMessage(content="2 + 2 = 4. \n\nOur conversation so far: We've discussed a simple math problem, and I've provided the answer to 2 + 2, which is 4.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 52, 'total_tokens': 93, 'completion_time': 0.152276657, 'completion_tokens_details': None, 'prompt_time': 0.003139385, 'prompt_tokens_details': None, 'queue_time': 0.052641825, 'total_time': 0.155416042}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ff167-b501-7fe3-bf18-4cdeb3ecf4e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 41, 'total_tokens': 93}), HumanMessage(content='what is 2+2?', additional_